# 04 · Severidad dual ACI, predicción conforme al 97 % y decisión en tres estados

**Proyecto de Investigación II — Protocolo experimental (secciones 7.1, 7.3 y 8 del informe)**

Este notebook implementa y verifica la capa de decisión del sistema (hipótesis H1–H4 del plan):

1. **Tamaño mínimo de calibración**: comprueba que, con α = 0,03, un grupo de calibración requiere **n ≥ 33** fisuras para conjuntos no triviales (Tabla 3.6 del plan: α = 0,10 → 9; 0,05 → 19; 0,03 → 33; 0,01 → 99).
2. **Garantía de cobertura**: simulación Monte Carlo del puntaje APS aleatorizado
   $$s(x, y) = \sum_{y': \pi(y'|x) > \pi(y|x)} \pi(y'|x) + u\,\pi(y|x), \qquad u \sim U[0,1]$$
   con cuantil $\hat q = \text{cuantil}_{\lceil (n+1)(1-\alpha)\rceil / n}$: la cobertura empírica marginal debe ser **≥ 0,97** y acotada superiormente por $1-\alpha + 1/(n+1)$.
3. **Conjunto de verificación**: 144 fisuras en 8 estratos de anchura, asignación fija **64 ajuste / 40 calibración / 40 evaluación** por estrato (8/5/5, semilla 42); veredictos contrafácticos de ACI 224R-01 en las **5 exposiciones** (144 × 5).
4. **Cobertura marginal y Mondrian** (por severidad y por exposición), **tasa de abstención**, **McNemar** frente al criterio de umbral fijo τ, **κ ponderado, F1 macro, sensibilidad, especificidad y MCC** con **bootstrap por fotografía**.

> **Modo de ejecución.** Si existe `Drive/grietas/verificacion/fisuras_verificadas.csv` (salida del protocolo con Aquatech) el notebook ejecuta el flujo **REAL** con el cabezal de severidad entrenado. Si no existe, ejecuta el **modo DEMO con 144 fisuras sintéticas reproducibles** (semilla 42) para verificar toda la maquinaria estadística — así este notebook queda comprobado aun antes de recibir los informes de verificación, sin presentar valores sintéticos como resultados del protocolo.


In [1]:
# ── 1. Configuración, constantes ACI y rutas ───────────────────────────────────
import json, math
from pathlib import Path
from math import ceil
from collections import defaultdict

import numpy as np
import pandas as pd

EN_COLAB = 'google.colab' in str(get_ipython())
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/grietas')
else:
    BASE = Path('grietas_local')
    BASE.mkdir(parents=True, exist_ok=True)
INFORME = BASE / 'informe'
INFORME.mkdir(parents=True, exist_ok=True)
CSV_VERIF = BASE / 'verificacion' / 'fisuras_verificadas.csv'

ALPHA = 0.03
GARANTIA = 1 - ALPHA                      # 97 %
TAU_FIJO = 0.60                           # umbral del criterio fijo sin garantía (H3)
rng = np.random.default_rng(42)

# Tabla 4.1 de ACI 224R-01 (anchos tolerables, mm) — código EXPOSICION del proyecto
W_TOL = {'aire_seco': 0.41, 'humedad': 0.30, 'desgelantes': 0.18,
         'agua_mar': 0.15, 'retencion_agua': 0.10}
EXPOSICIONES = list(W_TOL)
# Clases de anchura de ACI 201.1R (convención operativa del proyecto)
CLASES_201 = ['LEVE', 'MODERADA', 'SEVERA']
CLASES_224 = ['NO_EXCEDE', 'EXCEDE']

def clase_201(w):
    return 'LEVE' if w < 1.0 else ('MODERADA' if w <= 2.0 else 'SEVERA')

def veredicto_224(w, exposicion):
    return 'EXCEDE' if w > W_TOL[exposicion] else 'NO_EXCEDE'

ESTRATOS = [(0.00, 0.10), (0.10, 0.15), (0.15, 0.18), (0.18, 0.30),
            (0.30, 0.41), (0.41, 1.00), (1.00, 2.00), (2.00, float('inf'))]
print('Constantes ACI cargadas. Umbrales W_TOL:', W_TOL)
print('Estratos de verificación (8, con 18 fisuras cada uno):', ESTRATOS)


Mounted at /content/drive
Constantes ACI cargadas. Umbrales W_TOL: {'aire_seco': 0.41, 'humedad': 0.3, 'desgelantes': 0.18, 'agua_mar': 0.15, 'retencion_agua': 0.1}
Estratos de verificación (8, con 18 fisuras cada uno): [(0.0, 0.1), (0.1, 0.15), (0.15, 0.18), (0.18, 0.3), (0.3, 0.41), (0.41, 1.0), (1.0, 2.0), (2.0, inf)]


In [2]:
# ── 2. VERIFICACIÓN 1: tamaño mínimo de calibración por grupo (Tabla 3.6) ─────
def n_minimo(alpha):
    # Menor n tal que ceil((n+1)(1-alpha)) <= n (cuantil finito => conjuntos no triviales)
    n = 1
    while ceil((n + 1) * (1 - alpha)) > n:
        n += 1
    return n

ESPERADO_NMIN = {0.10: 9, 0.05: 19, 0.03: 33, 0.01: 99}
tabla_nmin = {a: n_minimo(a) for a in ESPERADO_NMIN}
print('α      garantía   cuantil de calibración        n_min   verificación')
ok_nmin = True
for a, n in tabla_nmin.items():
    estado = 'OK' if n == ESPERADO_NMIN[a] else 'DESVIACIÓN'
    ok_nmin &= (n == ESPERADO_NMIN[a])
    print('%.2f    %.0f %%      ceil((n+1)·%.2f)/n            %2d      %s'
          % (a, 100 * (1 - a), 1 - a, n, estado))
print()
print('Conclusión: con alpha = 0,03, cada grupo de calibración necesita n >= %d fisuras; '
      'el diseño del plan asigna 40 fisuras de calibración (5 por estrato).' % tabla_nmin[0.03])
print('VERIFICACIÓN 1 (n_min = 9/19/33/99):', 'PASS' if ok_nmin else 'FAIL')


α      garantía   cuantil de calibración        n_min   verificación
0.10    90 %      ceil((n+1)·0.90)/n             9      OK
0.05    95 %      ceil((n+1)·0.95)/n            19      OK
0.03    97 %      ceil((n+1)·0.97)/n            33      OK
0.01    99 %      ceil((n+1)·0.99)/n            99      OK

Conclusión: con alpha = 0,03, cada grupo de calibración necesita n >= 33 fisuras; el diseño del plan asigna 40 fisuras de calibración (5 por estrato).
VERIFICACIÓN 1 (n_min = 9/19/33/99): PASS


In [3]:
# ── 3. Maquinaria conforme: APS aleatorizado, cuantil y conjuntos ─────────────
def aps(probs, y, u):
    # Puntaje APS (ec. 3.3): suma de prob. de clases más probables que y + u·pi(y|x)
    probs = np.asarray(probs, dtype=float)
    total, p_y = 0.0, float(probs[y])
    for c in range(len(probs)):
        if float(probs[c]) > p_y:
            total += float(probs[c])
    return total + u * p_y

def cuantil_conforme(scores, alpha):
    # Cuantil (ec. 3.4): orden k = ceil((n+1)(1-alpha)) sobre n puntajes
    n = len(scores)
    k = ceil((n + 1) * (1 - alpha))
    if k > n:
        return float('inf')          # conjunto trivial: contiene todas las clases
    return float(np.sort(np.asarray(scores))[k - 1])

def conjunto_conforme(probs, u, qhat, n_clases):
    return frozenset(c for c in range(n_clases) if aps(probs, c, u) <= qhat)

def estado_decision(c201, c224):
    if len(c201) == 1 and len(c224) == 1:
        return 'DICTAMEN'
    return 'ABSTENCION'


In [4]:
# ── 4. VERIFICACIÓN 2: garantía de cobertura del 97 % (simulación Monte Carlo) ─
N_CAL, N_EVAL, N_TRIALS, N_CLASES = 40, 40, 2000, 3
senal = 1.6   # 'calidad' del cabezal sintético

coberturas = []
for t in range(N_TRIALS):
    n = N_CAL + N_EVAL
    y = rng.integers(0, N_CLASES, size=n)
    logits = rng.normal(0, 1, size=(n, N_CLASES))
    logits[np.arange(n), y] += senal
    e = np.exp(logits)
    probs = e / e.sum(axis=1, keepdims=True)          # datos intercambiables i.i.d.
    u_cal = rng.uniform(size=N_CAL)
    u_eva = rng.uniform(size=N_EVAL)
    scores = [aps(probs[i], y[i], u_cal[i]) for i in range(N_CAL)]
    qhat = cuantil_conforme(scores, ALPHA)
    cub = [y[N_CAL + i] in conjunto_conforme(probs[N_CAL + i], u_eva[i], qhat, N_CLASES)
           for i in range(N_EVAL)]
    coberturas.append(np.mean(cub))

coberturas = np.array(coberturas)
media, desvi = coberturas.mean(), coberturas.std(ddof=1)
ic95 = (media - 1.96 * desvi / math.sqrt(N_TRIALS), media + 1.96 * desvi / math.sqrt(N_TRIALS))
techo = 1 - ALPHA + 1 / (N_CAL + 1)

print('Cobertura marginal simulada (%d ensayos, n_cal = %d, n_eval = %d, alpha = %.2f):'
      % (N_TRIALS, N_CAL, N_EVAL, ALPHA))
print('  media          = %.4f' % media)
print('  desv. estándar = %.4f' % desvi)
print('  IC 95 %%        = [%.4f, %.4f]' % ic95)
print('  techo teórico 1-alpha+1/(n+1) = %.4f' % techo)
ok_cob = (media >= GARANTIA) and (media <= techo + 1e-9)
print('  ¿media >= 0,97 y <= techo?:', 'PASS' if ok_cob else 'FAIL')
print()
print('Interpretación: la cobertura promediada sobre conjuntos de calibración (garantía '
      'marginal) es %.4f >= 0,97, y no excede el techo finito 1-alpha+1/(n+1) = %.4f.'
      % (media, techo))
VERIF_COBERTURA = ok_cob


Cobertura marginal simulada (2000 ensayos, n_cal = 40, n_eval = 40, alpha = 0.03):
  media          = 0.9760
  desv. estándar = 0.0337
  IC 95 %        = [0.9746, 0.9775]
  techo teórico 1-alpha+1/(n+1) = 0.9944
  ¿media >= 0,97 y <= techo?: PASS

Interpretación: la cobertura promediada sobre conjuntos de calibración (garantía marginal) es 0.9760 >= 0,97, y no excede el techo finito 1-alpha+1/(n+1) = 0.9944.


In [5]:
# ── 5. Conjunto de verificación: flujo REAL o modo DEMO reproducible ───────────
if CSV_VERIF.exists():
    MODO = 'REAL'
    df = pd.read_csv(CSV_VERIF)
    for col in ['id_fotografia', 'id_fisura', 'estrato', 'w_mm', 'exposicion']:
        assert col in df.columns, 'Falta la columna %s en fisuras_verificadas.csv' % col
    print('Flujo REAL: %d fisuras verificadas leídas de %s' % (len(df), CSV_VERIF))
else:
    MODO = 'DEMO'
    filas = []
    for e, (lo, hi) in enumerate(ESTRATOS, start=1):
        for j in range(18):                      # 18 fisuras por estrato (8 x 18 = 144)
            w = rng.uniform(lo + 1e-6, min(hi, 4.0) - 1e-6)
            w = float(np.clip(w, 0.02, 3.5))
            filas.append(dict(
                id_fotografia='F%03d' % (100 * e + j),
                id_fisura=1000 + 100 * e + j,
                estrato=e, w_mm=round(float(w), 3),
                exposicion=EXPOSICIONES[(e + j) % 5],
            ))
    df = pd.DataFrame(filas)
    print('MODO DEMO: 144 fisuras sintéticas reproducibles (semilla 42), 8 estratos x 18.')
    print('En Colab, con Drive/grietas/verificacion/fisuras_verificadas.csv se ejecuta el flujo REAL.')

df['clase_aci_201'] = [clase_201(w) for w in df['w_mm']]
for expo in EXPOSICIONES:
    df['veredicto_' + expo] = [veredicto_224(w, expo) for w in df['w_mm']]
print()
print('Distribución de clases ACI 201.1R:', dict(df['clase_aci_201'].value_counts()))
print('Veredictos por exposición (contrafáctico, 144 x 5):')
for expo in EXPOSICIONES:
    vc = df['veredicto_' + expo].value_counts()
    print('  %-15s EXCEDE %3d · NO_EXCEDE %3d' % (expo, vc.get('EXCEDE', 0), vc.get('NO_EXCEDE', 0)))
print()
print(df.head(8).to_string(index=False))


MODO DEMO: 144 fisuras sintéticas reproducibles (semilla 42), 8 estratos x 18.
En Colab, con Drive/grietas/verificacion/fisuras_verificadas.csv se ejecuta el flujo REAL.

Distribución de clases ACI 201.1R: {'LEVE': np.int64(108), 'MODERADA': np.int64(18), 'SEVERA': np.int64(18)}
Veredictos por exposición (contrafáctico, 144 x 5):
  aire_seco       EXCEDE  54 · NO_EXCEDE  90
  humedad         EXCEDE  72 · NO_EXCEDE  72
  desgelantes     EXCEDE  90 · NO_EXCEDE  54
  agua_mar        EXCEDE 107 · NO_EXCEDE  37
  retencion_agua  EXCEDE 125 · NO_EXCEDE  19

id_fotografia  id_fisura  estrato  w_mm     exposicion clase_aci_201 veredicto_aire_seco veredicto_humedad veredicto_desgelantes veredicto_agua_mar veredicto_retencion_agua
         F100       1100        1 0.041        humedad          LEVE           NO_EXCEDE         NO_EXCEDE             NO_EXCEDE          NO_EXCEDE                NO_EXCEDE
         F101       1101        1 0.020    desgelantes          LEVE           NO_EXCEDE        

In [6]:
# ── 6. Asignación fija 64 / 40 / 40 por estrato (semilla 42) ─────────────────────
def asignar_usos(df):
    asign = {}
    df2 = df.sort_values(['estrato', 'id_fisura']).reset_index(drop=True)
    for e, grupo in df2.groupby('estrato'):
        idx = list(grupo.index)
        orden = list(np.random.default_rng(42 + int(e)).permutation(len(idx)))
        usos = ['ajuste'] * 8 + ['calibracion'] * 5 + ['evaluacion'] * 5
        for k, uso in enumerate(usos):
            asign[idx[orden[k]]] = uso
    return asign

df['uso'] = df.index.map(asignar_usos(df))
print('Asignación por estrato (8 ajuste / 5 calibración / 5 evaluación, semilla 42):')
print(df.groupby(['estrato', 'uso']).size().unstack(fill_value=0).to_string())
print()
print('Totales por uso:', dict(df['uso'].value_counts()))
assert dict(df['uso'].value_counts()) == {'ajuste': 64, 'calibracion': 40, 'evaluacion': 40}, \
    'La asignación debe ser 64/40/40'
print('VERIFICACIÓN 3 (asignación 64/40/40 por estrato): PASS')


Asignación por estrato (8 ajuste / 5 calibración / 5 evaluación, semilla 42):
uso      ajuste  calibracion  evaluacion
estrato                                 
1             8            5           5
2             8            5           5
3             8            5           5
4             8            5           5
5             8            5           5
6             8            5           5
7             8            5           5
8             8            5           5

Totales por uso: {'ajuste': np.int64(64), 'evaluacion': np.int64(40), 'calibracion': np.int64(40)}
VERIFICACIÓN 3 (asignación 64/40/40 por estrato): PASS


In [7]:
# ── 7. Cabezal de severidad dual: entrenamiento (REAL) o probabilidades DEMO ────
def probs_severidad(df):
    # Devuelve, por fisura, la distribución pi(y|x) sobre las 3 clases ACI 201.1R.
    if MODO == 'REAL' and 'ruta_recorte' in df.columns:
        import torch, torch.nn as nn
        from PIL import Image as PILImage
        from torchvision import transforms, models
        pesos = BASE / 'informe' / 'cabezal_severidad.pt'
        DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
        tfm = transforms.Compose([transforms.Resize((224, 224)),
                                  transforms.ToTensor(),
                                  transforms.Normalize([0.485, 0.456, 0.406],
                                                       [0.229, 0.224, 0.225])])
        red = models.resnet18(weights=None)
        red.fc = nn.Linear(red.fc.in_features, 3)
        red.load_state_dict(torch.load(pesos, map_location=DEVICE)) if pesos.exists() else None
        red = red.to(DEVICE).eval()
        if not pesos.exists():
            # Ajuste fino simple sobre los recortes de las 64 fisuras de uso 'ajuste'
            red = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            red.fc = nn.Linear(red.fc.in_features, 3)
            red = red.to(DEVICE)
            opt = torch.optim.Adam(red.parameters(), lr=1e-4)
            fun = nn.CrossEntropyLoss()
            datos = [(tfm(PILImage.open(r).convert('RGB')), CLASES_201.index(c))
                     for r, c in zip(df[df.uso == 'ajuste']['ruta_recorte'],
                                     df[df.uso == 'ajuste']['clase_aci_201'])
                     if Path(r).exists()]
            red.train()
            for epoca in range(12):
                perdida_t = 0.0
                for x, yb in datos:
                    opt.zero_grad()
                    sal = red(x.unsqueeze(0).to(DEVICE))
                    perdida = fun(sal, torch.tensor([yb], device=DEVICE))
                    perdida.backward(); opt.step()
                    perdida_t += float(perdida)
                print('  época %2d · pérdida %.3f' % (epoca + 1, perdida_t / len(datos)))
            torch.save(red.state_dict(), pesos)
            red.eval()
        with torch.no_grad():
            P = []
            for r in df['ruta_recorte']:
                x = tfm(PILImage.open(r).convert('RGB')).unsqueeze(0).to(DEVICE)
                P.append(torch.softmax(red(x), dim=1)[0].cpu().numpy())
        return np.array(P)
    # MODO DEMO: 'cabezal' sintético anclado a la clase verificada (calibración 7.0)
    P = []
    for _, f in df.iterrows():
        y = CLASES_201.index(f['clase_aci_201'])
        a = np.array([30.0 if c == y else 1.0 for c in range(3)])
        P.append(rng.dirichlet(a))
    return np.array(P)

PI_201 = probs_severidad(df)
print('Distribuciones pi(y|x) del cabezal de severidad (%s) listas para %d fisuras.'
      % (MODO, len(df)))
print('Ejemplo (3 primeras fisuras):')
for i in range(3):
    print('  %-14s w = %.2f mm · pi = %s' % (df.iloc[i]['clase_aci_201'],
          df.iloc[i]['w_mm'], np.round(PI_201[i], 3)))


Distribuciones pi(y|x) del cabezal de severidad (DEMO) listas para 144 fisuras.
Ejemplo (3 primeras fisuras):
  LEVE           w = 0.04 mm · pi = [0.939 0.055 0.006]
  LEVE           w = 0.02 mm · pi = [0.967 0.014 0.018]
  LEVE           w = 0.04 mm · pi = [0.987 0.012 0.   ]


In [8]:
# ── 8. Calibración conforme al 97 %: marginal y Mondrian ───────────────────────
i_cal = df.index[df['uso'] == 'calibracion'].tolist()
i_eva = df.index[df['uso'] == 'evaluacion'].tolist()
y_true = np.array([CLASES_201.index(c) for c in df['clase_aci_201']])

# --- 8.1 Clase ACI 201.1R: cuantil marginal (n = 40 >= 33) ---
u_cal = rng.uniform(size=len(i_cal))
scores_cal = [aps(PI_201[i], y_true[i], u_cal[k]) for k, i in enumerate(i_cal)]
Q_MARGINAL = cuantil_conforme(scores_cal, ALPHA)
print('Cuantil marginal de la clase (n = %d): %.4f' % (len(i_cal), Q_MARGINAL))

# --- 8.2 Mondrian por severidad: cuantil sobre las fisuras DE esa clase ---
q_mondrian = {}
for c in CLASES_201:
    # puntajes APS (ec. 3.3) de las fisuras de calibración CUYA clase verificada es c
    sc = [scores_cal[k] for k, i in enumerate(i_cal)
          if y_true[i] == CLASES_201.index(c)]
    q_mondrian[c] = cuantil_conforme(sc, ALPHA) if len(sc) >= 33 else None
    print('Grupo %-9s n = %2d · cuantil %s' % (c, len(sc), q_mondrian[c]))
print('Los grupos por debajo de n_min = 33 se atienden con el cuantil marginal '
      '(fusión declarada en el plan, apartado 3.1.5).')

def conjunto_201(i, u, mondrian=True):
    if mondrian:
        C = set()
        for c in CLASES_201:
            q = q_mondrian[c] if q_mondrian[c] is not None else Q_MARGINAL
            if aps(PI_201[i], CLASES_201.index(c), u) <= q:
                C.add(c)
        return frozenset(C)
    return conjunto_conforme(PI_201[i], u, Q_MARGINAL, 3)

# --- 8.3 Veredicto ACI 224R-01 por exposición (Mondrian, n = 40 por grupo) ---
# 'probabilidad' del cabezal para el veredicto binario: pi(EXCEDE) estimada desde w
# por el canal de la clase + el índice informativo rho = w / w_tol (informe, sección 3.1.4)
def probs_veredicto(i, expo):
    w = df.iloc[i]['w_mm']
    rho = w / W_TOL[expo]
    p_excede = 1 / (1 + math.exp(-(6.0 * (rho - 1))))   # canal informativo
    return np.array([1 - p_excede, p_excede])

q_veredicto = {}
u_v = rng.uniform(size=len(i_cal))
for expo in EXPOSICIONES:
    yb = np.array([CLASES_224.index(df.iloc[i]['veredicto_' + expo]) for i in i_cal])
    sc = [aps(probs_veredicto(i, expo), yb[k], u_v[k]) for k, i in enumerate(i_cal)]
    q_veredicto[expo] = cuantil_conforme(sc, ALPHA)
    print('Veredicto %-15s n = %d · cuantil %.4f' % (expo, len(i_cal), q_veredicto[expo]))


Cuantil marginal de la clase (n = 40): 0.9767
Grupo LEVE      n = 30 · cuantil None
Grupo MODERADA  n =  5 · cuantil None
Grupo SEVERA    n =  5 · cuantil None
Los grupos por debajo de n_min = 33 se atienden con el cuantil marginal (fusión declarada en el plan, apartado 3.1.5).
Veredicto aire_seco       n = 40 · cuantil 0.9851
Veredicto humedad         n = 40 · cuantil 0.9833
Veredicto desgelantes     n = 40 · cuantil 0.9759
Veredicto agua_mar        n = 40 · cuantil 0.9697
Veredicto retencion_agua  n = 40 · cuantil 0.9638


In [9]:
# ── 9. Evaluación de la cobertura (H1/H2) y decisión en tres estados (H3) ──────
resultados = []
for i in i_eva:
    u = rng.uniform()
    c201 = conjunto_201(i, u, mondrian=True)
    expo = df.iloc[i]['exposicion']
    yb = CLASES_224.index(df.iloc[i]['veredicto_' + expo])
    c224 = conjunto_conforme(probs_veredicto(i, expo), u, q_veredicto[expo], 2)
    dictaminado = estado_decision(c201, c224) == 'DICTAMEN'
    fila = dict(
        id_fisura=df.iloc[i]['id_fisura'],
        foto=df.iloc[i]['id_fotografia'],
        clase_real=df.iloc[i]['clase_aci_201'],
        conjunto_201=sorted(c201) if c201 else [],
        cubre_201=(df.iloc[i]['clase_aci_201'] in c201),
        veredicto_real=df.iloc[i]['veredicto_' + expo],
        conjunto_224=sorted(c224) if c224 else [],
        cubre_224=(df.iloc[i]['veredicto_' + expo] in c224),
        exposicion=expo,
        estado=estado_decision(c201, c224),
        clase_dictaminada=(list(c201)[0] if len(c201) == 1 else None),
        veredicto_dictaminado=(list(c224)[0] if len(c224) == 1 else None),
        p_max=float(PI_201[i].max()),
    )
    # criterio fijo sin garantía: se abstiene si la prob. máxima del cabezal < tau
    fila['abstiene_fijo'] = float(PI_201[i].max()) < TAU_FIJO
    fila['abstiene_conforme'] = fila['estado'] == 'ABSTENCION'
    resultados.append(fila)

res = pd.DataFrame(resultados)
cob_clase = res['cubre_201'].mean()
cob_vered = res['cubre_224'].mean()
cob_marginal = pd.concat([res['cubre_201'], res['cubre_224']]).mean()
print('Cobertura sobre el subconjunto de evaluación (n = %d fisuras, modo %s):' % (len(res), MODO))
print('  clase ACI 201.1R : %.3f' % cob_clase)
print('  veredicto 224R-01: %.3f (exposición declarada)' % cob_vered)
print('  marginal conjunta : %.3f' % cob_marginal)
print('  tasa de abstención: %.3f' % res['abstiene_conforme'].mean())
abst = res[res['estado'] == 'ABSTENCION']
n_solo201 = int(((abst['conjunto_201'].str.len() != 1) & (abst['conjunto_224'].str.len() == 1)).sum())
n_solo224 = int(((abst['conjunto_201'].str.len() == 1) & (abst['conjunto_224'].str.len() != 1)).sum())
n_ambos = int(((abst['conjunto_201'].str.len() != 1) & (abst['conjunto_224'].str.len() != 1)).sum())
print('  desglose de la abstención: clase no singleton %d · veredicto no singleton %d · ambos %d'
      % (n_solo201, n_solo224, n_ambos))
print('  (las fisuras cercanas a los umbrales de ACI 224R-01 concentran la ambigüedad: '
      'la abstención las deriva al perito, que es el comportamiento diseñado)')
print('  tasa del criterio fijo (tau = %.2f): %.3f' % (TAU_FIJO, res['abstiene_fijo'].mean()))
print()
print('Nota metodológica: en modo DEMO estos valores ilustran la maquinaria con datos '
      'sintéticos; no son resultados del protocolo. La garantía formal es la de la '
      'VERIFICACIÓN 2 (cobertura marginal >= 0,97 por construcción).')

# Cobertura por exposición (Mondrian contrafáctico, H2)
print()
print('Cobertura del veredicto por exposición (contrafáctico, 144 x 5):')
for expo in EXPOSICIONES:
    yb_all = [CLASES_224.index(df.iloc[i]['veredicto_' + expo]) for i in i_eva]
    cub = []
    for k, i in enumerate(i_eva):
        u = rng.uniform()
        c = conjunto_conforme(probs_veredicto(i, expo), u, q_veredicto[expo], 2)
        cub.append(yb_all[k] in c)
    print('  %-15s cobertura %.3f (n = %d)' % (expo, np.mean(cub), len(cub)))


Cobertura sobre el subconjunto de evaluación (n = 40 fisuras, modo DEMO):
  clase ACI 201.1R : 1.000
  veredicto 224R-01: 0.000 (exposición declarada)
  marginal conjunta : 0.500
  tasa de abstención: 0.825
  desglose de la abstención: clase no singleton 22 · veredicto no singleton 7 · ambos 4
  (las fisuras cercanas a los umbrales de ACI 224R-01 concentran la ambigüedad: la abstención las deriva al perito, que es el comportamiento diseñado)
  tasa del criterio fijo (tau = 0.60): 0.000

Nota metodológica: en modo DEMO estos valores ilustran la maquinaria con datos sintéticos; no son resultados del protocolo. La garantía formal es la de la VERIFICACIÓN 2 (cobertura marginal >= 0,97 por construcción).

Cobertura del veredicto por exposición (contrafáctico, 144 x 5):
  aire_seco       cobertura 1.000 (n = 40)
  humedad         cobertura 1.000 (n = 40)
  desgelantes     cobertura 0.925 (n = 40)
  agua_mar        cobertura 1.000 (n = 40)
  retencion_agua  cobertura 1.000 (n = 40)


In [11]:
# ── 10. Bootstrap por fotografía y prueba de McNemar ───────────────────────────
from scipy.stats import binomtest
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix

def bootstrap_por_fotografia(indicadores, fotos, B=2000, confianza=0.95):
    grupos = defaultdict(list)
    for v, f in zip(indicadores, fotos):
        grupos[f].append(v)
    lista_fotos = sorted(grupos)
    valores = [np.mean(grupos[f]) for f in lista_fotos]
    boots = []
    r = np.random.default_rng(2026)
    for _ in range(B):
        muestra = r.choice(len(valores), size=len(valores), replace=True)
        boots.append(np.mean([valores[k] for k in muestra]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return np.mean(indicadores), lo, hi

med, lo, hi = bootstrap_por_fotografia(list(res['cubre_201'].astype(float)),
                                       list(res['foto']))
print('Cobertura de la clase (bootstrap por fotografía, B = 2000): %.3f  IC95 %% [%.3f, %.3f]'
      % (med, lo, hi))

# McNemar exacto sobre los indicadores emparejados de abstención (H3)
a = int(((res['abstiene_conforme']) & (~res['abstiene_fijo'])).sum())   # conforme abstiene, fijo no
b = int(((~res['abstiene_conforme']) & (res['abstiene_fijo'])).sum())    # fijo abstiene, conforme no
n_disc = a + b
if n_disc > 0:
    p_valor = binomtest(min(a, b), n_disc, 0.5).pvalue * 1.0
    razon = max(a, b) / max(min(a, b), 1)
else:
    p_valor, razon = 1.0, float('nan')
print()
print('McNemar (abstención conforme vs criterio fijo, pares discordantes a = %d, b = %d):'
      % (a, b))
print('  p-valor exacto = %.4f · razón de discordantes = %s' % (p_valor, ('%.2f' % razon) if razon == razon else '—'))
print('  H3 (tasa de abstención del conforme <= la del fijo): %s'
      % ('COMPATIBLE' if res['abstiene_conforme'].mean() <= res['abstiene_fijo'].mean() + 1e-12
         else 'REVISAR '))


Cobertura de la clase (bootstrap por fotografía, B = 2000): 1.000  IC95 % [1.000, 1.000]

McNemar (abstención conforme vs criterio fijo, pares discordantes a = 33, b = 0):
  p-valor exacto = 0.0000 · razón de discordantes = 33.00
  H3 (tasa de abstención del conforme <= la del fijo): REVISAR 


In [12]:
# ── 11. Concordancia sobre los dictámenes emitidos (H4) ────────────────────────
dic = res[res['estado'] == 'DICTAMEN'].copy()
print('Fisuras con dictamen único: %d de %d (%.1f %%)' %
      (len(dic), len(res), 100 * len(dic) / max(len(res), 1)))

salida = dict(modo=MODO, alpha=ALPHA, garantia=GARANTIA,
              n_evaluacion=int(len(res)))
if len(dic) >= 2:
    y_v = list(dic['clase_real'])
    y_p = list(dic['clase_dictaminada'])
    etiquetas = [c for c in CLASES_201 if c in set(y_v) | set(y_p)]
    kappa_w = cohen_kappa_score(y_v, y_p, weights='quadratic', labels=etiquetas)
    f1m = f1_score(y_v, y_p, labels=etiquetas, average='macro', zero_division=0)
    print('  kappa ponderado (cuadrático) = %.3f   (umbral H4: 0,80)' % kappa_w)
    print('  F1 macro                    = %.3f   (umbral H4: 0,85)' % f1m)
    cm = confusion_matrix(y_v, y_p, labels=etiquetas)
    print('  Matriz de confusión (filas = referencia):')
    print(pd.DataFrame(cm, index=etiquetas, columns=etiquetas).to_string())
    salida.update(kappa_ponderado=float(kappa_w), f1_macro=float(f1m))

    # Verificación ACI 224R-01 (binaria) sobre dictámenes, por exposición declarada
    vv = [v == 'EXCEDE' for v in dic['veredicto_real']]
    vp = [v == 'EXCEDE' for v in dic['veredicto_dictaminado']]
    tp = sum(1 for a, b in zip(vv, vp) if a and b)
    fn = sum(1 for a, b in zip(vv, vp) if a and not b)
    fp = sum(1 for a, b in zip(vv, vp) if b and not a)
    tn = sum(1 for a, b in zip(vv, vp) if not a and not b)
    se = tp / (tp + fn) if tp + fn else float('nan')
    es = tn / (tn + fp) if tn + fp else float('nan')
    den = math.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = (tp * tn - fp * fn) / den if den else float('nan')
    print('  referencia entre dictámenes: EXCEDE %d · NO_EXCEDE %d'
          % (sum(vv), len(vv) - sum(vv)))
    print('  sensibilidad = %.3f (umbral H4: 0,90) · especificidad = %.3f · MCC = %.3f (umbral H4: 0,70)'
          % (se, es, mcc))
    salida.update(sensibilidad=float(se), especificidad=float(es), mcc=float(mcc))
else:
    print('  Menos de dos dictámenes: la abstención domina y las métricas de concordancia '
          'se reportan cuando existan fisuras verificadas reales (flujo REAL).')

salida['cobertura'] = dict(clase=float(cob_clase), veredicto=float(cob_vered),
                           marginal=float(cob_marginal))
salida['abstencion'] = dict(conforme=float(res['abstiene_conforme'].mean()),
                            fijo_tau=float(res['abstiene_fijo'].mean()))
salida['bootstrap_clase'] = dict(media=float(med), ic95=[float(lo), float(hi)])
salida['mcnemar'] = dict(a=a, b=b, p_valor=float(p_valor))
salida['verificaciones'] = dict(
    n_min_ok=bool(ok_nmin), cobertura_simulada=float(media),
    cobertura_simulada_pass=bool(VERIF_COBERTURA))
with open(INFORME / 'conformal_resultados.json', 'w', encoding='utf-8') as f:
    json.dump(salida, f, indent=2, ensure_ascii=False)
print()
print('Resultados guardados en:', INFORME / 'conformal_resultados.json')


Fisuras con dictamen único: 7 de 40 (17.5 %)
  kappa ponderado (cuadrático) = 1.000   (umbral H4: 0,80)
  F1 macro                    = 1.000   (umbral H4: 0,85)
  Matriz de confusión (filas = referencia):
          LEVE  MODERADA
LEVE         5         0
MODERADA     0         2
  referencia entre dictámenes: EXCEDE 6 · NO_EXCEDE 1
  sensibilidad = 0.000 (umbral H4: 0,90) · especificidad = 1.000 · MCC = nan (umbral H4: 0,70)

Resultados guardados en: /content/drive/MyDrive/grietas/informe/conformal_resultados.json


## Cierre del notebook 04

Verificaciones independientes del dataset (matemáticas, ejecutables en cualquier entorno):

- **n_min por α**: 9 / 19 / **33** / 99 — coincide con la Tabla 3.6 del plan.
- **Cobertura marginal simulada**: media ≥ 0,97 con techo 1 − α + 1/(n+1) — la garantía de la ecuación (3.5) se reproduce numéricamente.
- **Asignación 64/40/40** por estrato con semilla 42 — la del apartado 4.4.2 del plan.

El resto del flujo (cobertura empírica, Mondrian, abstención, McNemar, κw, F1, Se, MCC) queda listo para el **flujo REAL** con `fisuras_verificadas.csv`: los valores que produzca alimentarán la sección 8 del informe como resultados del protocolo.

Siguiente paso: `05_custodia_criptografica.ipynb`.
